# QuantIQ | Phase 2 — Market Analysis

**Phase:** 2 | **Weeks:** 4–6 | **Deadline:** 14 June 2026

**Purpose:** Individual stock analysis for 12 NIFTY50 stocks — technical indicators, fundamentals, and cross-sector correlation.

---

## Team

| Handle | Stock          | Sector                |
|--------|----------------|-----------------------|
| AJ     | RELIANCE.NS    | Energy / Conglomerate |
| AV     | TCS.NS         | IT                    |
| AK     | INFY.NS        | IT                    |
| SS     | HCLTECH.NS     | IT                    |
| AR     | HDFCBANK.NS    | Banking               |
| EB     | ICICIBANK.NS   | Banking               |
| ShS    | AXISBANK.NS    | Banking               |
| RS     | TVSMOTOR.NS         | Auto                  |
| GT     | M&M.NS         | Auto                  |
| RT     | LT.NS          | Infrastructure        |
| NS     | TITAN.NS       | Consumer              |
| SmS    | APOLLOMICRO.NS | Defence               |

**Section 13 — Correlation Heatmap:** RS + GT

---

*Created by RS (Project Lead). Do not modify header cells or shared helper cells.*

In [1]:
import warnings
warnings.filterwarnings("ignore")

import sys
import os
sys.path.insert(0, os.path.abspath(".."))

import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
import plotly.io as pio
from plotly.subplots import make_subplots
import yfinance as yf
from ta.trend import EMAIndicator, MACD
from ta.momentum import RSIIndicator
from ta.volatility import AverageTrueRange
from src.data import fetch_ohlc

pio.templates["plotly_dark"].layout.paper_bgcolor = "#252526"
pio.templates["plotly_dark"].layout.plot_bgcolor  = "#1F2531"
pio.templates.default = "plotly_dark"

print("Imports OK")

Imports OK


In [2]:
TICKERS = {
    "AJ":  "RELIANCE.NS",
    "AV":  "TCS.NS",
    "AK":  "INFY.NS",
    "SS":  "HCLTECH.NS",
    "AR":  "HDFCBANK.NS",
    "EB":  "ICICIBANK.NS",
    "ShS": "AXISBANK.NS",
    "RS":  "TVSMOTOR.NS",
    "GT":  "M&M.NS",
    "RT":  "LT.NS",
    "NS":  "TITAN.NS",
    "SmS": "APOLLOMICRO.NS",
}

PERIOD   = "1y"
INTERVAL = "1d"

print(f"Config loaded — {len(TICKERS)} tickers, period={PERIOD}, interval={INTERVAL}")

Config loaded — 12 tickers, period=1y, interval=1d


In [3]:
def fetch_stock(
    ticker: str,
    period: str = PERIOD,
    interval: str = INTERVAL,
) -> pd.DataFrame:
    """Fetch OHLCV for a .NS ticker via fetch_ohlc.

    Args:
        ticker (str): NSE ticker with .NS suffix (e.g. "RELIANCE.NS").
        period (str): yfinance period string. Default "1y".
        interval (str): yfinance interval string. Default "1d".

    Returns:
        pd.DataFrame: OHLCV DataFrame indexed by datetime, NaN rows dropped.

    Raises:
        AssertionError: If ticker does not end with .NS.
        ValueError: If no data returned for ticker.
    """
    assert ticker.endswith(".NS"), f"Ticker must have .NS suffix, got: {ticker}"
    df = fetch_ohlc(ticker, period=period, interval=interval, use_cache=True)
    if df is None or df.empty:
        raise ValueError(f"No data returned for {ticker}")
    return df.dropna()

## Energy

## Section 1 — RELIANCE.NS | AJ (Energy / Conglomerate)

In [ ]:
# ── 1a. Fetch ──────────────────────────────────────────────────────────────
ticker = TICKERS["AJ"]
df     = fetch_stock(ticker)
print(f"{ticker} | {df.shape[0]} rows | {df.index[0].date()} → {df.index[-1].date()}")
df.tail()

In [ ]:
# ── 1b. Summary Statistics ──────────────────────────────────────────────────
print(f"Shape     : {df.shape}")
print(f"Date range: {df.index[0].date()} → {df.index[-1].date()}")
df.describe().round(2)

In [ ]:
# ── 1c. Price: Candlestick + EMA20 / EMA50 ────────────────────────────────
ema20 = EMAIndicator(close=df["Close"], window=20).ema_indicator()
ema50 = EMAIndicator(close=df["Close"], window=50).ema_indicator()

fig = make_subplots(rows=1, cols=1)
fig.add_trace(go.Candlestick(
    x=df.index, open=df["Open"], high=df["High"],
    low=df["Low"], close=df["Close"], name="OHLC",
))
fig.add_trace(go.Scatter(x=df.index, y=ema20, name="EMA20", line=dict(color="orange", width=1.5)))
fig.add_trace(go.Scatter(x=df.index, y=ema50, name="EMA50", line=dict(color="#4A90D9", width=1.5)))
fig.update_layout(
    title=f"{ticker} — Price + EMA20/50",
    xaxis_rangeslider_visible=False,
    height=500,
)
fig.show()

In [ ]:
# ── 1d. MACD (12/26/9) ────────────────────────────────────────────────────
macd_ind    = MACD(close=df["Close"], window_slow=26, window_fast=12, window_sign=9)
macd_line   = macd_ind.macd()
signal_line = macd_ind.macd_signal()
histogram   = macd_ind.macd_diff()

fig = make_subplots(
    rows=2, cols=1, shared_xaxes=True,
    row_heights=[0.6, 0.4],
    subplot_titles=[f"{ticker} — Close", "MACD (12/26/9)"],
)
fig.add_trace(go.Scatter(x=df.index, y=df["Close"], name="Close",  line=dict(color="white")),  row=1, col=1)
fig.add_trace(go.Scatter(x=df.index, y=macd_line,   name="MACD",   line=dict(color="cyan")),   row=2, col=1)
fig.add_trace(go.Scatter(x=df.index, y=signal_line, name="Signal", line=dict(color="orange")), row=2, col=1)
fig.add_trace(go.Bar(    x=df.index, y=histogram,   name="Hist",   marker_color="white"),       row=2, col=1)
fig.update_layout(height=600, xaxis_rangeslider_visible=False)
fig.show()

In [ ]:
# ── 1e. RSI (14) ───────────────────────────────────────────────────────────
rsi = RSIIndicator(close=df["Close"], window=14).rsi()

fig = go.Figure()
fig.add_trace(go.Scatter(x=df.index, y=rsi, name="RSI(14)", line=dict(color="mediumpurple", width=1.5)))
fig.add_hline(y=70, line_dash="dash", line_color="red",       annotation_text="Overbought (70)")
fig.add_hline(y=30, line_dash="dash", line_color="limegreen", annotation_text="Oversold (30)")
fig.update_layout(
    title=f"{ticker} — RSI (14)",
    yaxis=dict(range=[0, 100]),
    height=350,
)
fig.show()

In [ ]:
# ── 1f. Volume ──────────────────────────────────────────────────────────────
vol_avg = df["Volume"].rolling(20).mean()

fig = go.Figure()
fig.add_trace(go.Bar(   x=df.index, y=df["Volume"], name="Volume",  marker_color="white", opacity=0.6))
fig.add_trace(go.Scatter(x=df.index, y=vol_avg,     name="20d Avg", line=dict(color="orange", width=1.5)))
fig.update_layout(title=f"{ticker} — Volume + 20-day Avg", height=350)
fig.show()

In [ ]:
# ── 1g. Fundamentals ─────────────────────────────────────────────────────────
ticker = TICKERS["AJ"]
info   = yf.Ticker(ticker).info

pe    = info.get("trailingPE");     pe_str  = f"{pe:.2f}"    if pe    is not None else "N/A"
eps_g = info.get("earningsGrowth"); eps_str = f"{eps_g:.2%}" if eps_g is not None else "N/A"
de    = info.get("debtToEquity");   de_str  = f"{de:.2f}"    if de    is not None else "N/A"
cash  = info.get("totalCash");      c_str   = f"{cash:,.0f}" if cash  is not None else "N/A"
debt  = info.get("totalDebt");      d_str   = f"{debt:,.0f}" if debt  is not None else "N/A"
icr   = info.get("currentRatio");   icr_str = f"{icr:.2f}"   if icr   is not None else "N/A"
peg   = info.get("pegRatio");       peg_str = f"{peg:.2f}"   if peg   is not None else "N/A"
div_y = info.get("dividendYield");  div_str = f"{div_y:.2%}" if div_y is not None else "N/A"

print(f"  P/E Ratio        : {pe_str}")
print(f"  EPS Growth (TTM) : {eps_str}")
print(f"  Debt / Equity    : {de_str}")
print(f"  Total Cash       : ₹{c_str}")
print(f"  Total Debt       : ₹{d_str}")
print(f"  Current Ratio    : {icr_str}")
print(f"  PEG Ratio        : {peg_str}")
print(f"  Dividend Yield   : {div_str}")

### Observations

_TODO (AJ): Write 3–5 paragraphs on what the technical and fundamental data reveals about RELIANCE.NS. Consider: trend direction, momentum signals (RSI / MACD), valuation (P/E, PEG), and financial health (D/E, cash position)._

## IT

## Section 2 — TCS.NS | AV (IT)

In [ ]:
# ── 2a. Fetch ──────────────────────────────────────────────────────────────
ticker = TICKERS["AV"]
df     = fetch_stock(ticker)
print(f"{ticker} | {df.shape[0]} rows | {df.index[0].date()} → {df.index[-1].date()}")
df.tail()

In [ ]:
# ── 2b. Summary Statistics ──────────────────────────────────────────────────
print(f"Shape     : {df.shape}")
print(f"Date range: {df.index[0].date()} → {df.index[-1].date()}")
df.describe().round(2)

In [ ]:
# ── 2c. Price: Candlestick + EMA20 / EMA50 ────────────────────────────────
ema20 = EMAIndicator(close=df["Close"], window=20).ema_indicator()
ema50 = EMAIndicator(close=df["Close"], window=50).ema_indicator()

fig = make_subplots(rows=1, cols=1)
fig.add_trace(go.Candlestick(
    x=df.index, open=df["Open"], high=df["High"],
    low=df["Low"], close=df["Close"], name="OHLC",
))
fig.add_trace(go.Scatter(x=df.index, y=ema20, name="EMA20", line=dict(color="orange", width=1.5)))
fig.add_trace(go.Scatter(x=df.index, y=ema50, name="EMA50", line=dict(color="#4A90D9", width=1.5)))
fig.update_layout(
    title=f"{ticker} — Price + EMA20/50",
    xaxis_rangeslider_visible=False,
    height=500,
)
fig.show()

In [ ]:
# ── 2d. MACD (12/26/9) ────────────────────────────────────────────────────
macd_ind    = MACD(close=df["Close"], window_slow=26, window_fast=12, window_sign=9)
macd_line   = macd_ind.macd()
signal_line = macd_ind.macd_signal()
histogram   = macd_ind.macd_diff()

fig = make_subplots(
    rows=2, cols=1, shared_xaxes=True,
    row_heights=[0.6, 0.4],
    subplot_titles=[f"{ticker} — Close", "MACD (12/26/9)"],
)
fig.add_trace(go.Scatter(x=df.index, y=df["Close"], name="Close",  line=dict(color="white")),  row=1, col=1)
fig.add_trace(go.Scatter(x=df.index, y=macd_line,   name="MACD",   line=dict(color="cyan")),   row=2, col=1)
fig.add_trace(go.Scatter(x=df.index, y=signal_line, name="Signal", line=dict(color="orange")), row=2, col=1)
fig.add_trace(go.Bar(    x=df.index, y=histogram,   name="Hist",   marker_color="white"),       row=2, col=1)
fig.update_layout(height=600, xaxis_rangeslider_visible=False)
fig.show()

In [ ]:
# ── 2e. RSI (14) ───────────────────────────────────────────────────────────
rsi = RSIIndicator(close=df["Close"], window=14).rsi()

fig = go.Figure()
fig.add_trace(go.Scatter(x=df.index, y=rsi, name="RSI(14)", line=dict(color="mediumpurple", width=1.5)))
fig.add_hline(y=70, line_dash="dash", line_color="red",       annotation_text="Overbought (70)")
fig.add_hline(y=30, line_dash="dash", line_color="limegreen", annotation_text="Oversold (30)")
fig.update_layout(
    title=f"{ticker} — RSI (14)",
    yaxis=dict(range=[0, 100]),
    height=350,
)
fig.show()

In [ ]:
# ── 2f. Volume ──────────────────────────────────────────────────────────────
vol_avg = df["Volume"].rolling(20).mean()

fig = go.Figure()
fig.add_trace(go.Bar(   x=df.index, y=df["Volume"], name="Volume",  marker_color="white", opacity=0.6))
fig.add_trace(go.Scatter(x=df.index, y=vol_avg,     name="20d Avg", line=dict(color="orange", width=1.5)))
fig.update_layout(title=f"{ticker} — Volume + 20-day Avg", height=350)
fig.show()

In [ ]:
# ── 2g. Fundamentals ─────────────────────────────────────────────────────────
ticker = TICKERS["AV"]
info   = yf.Ticker(ticker).info

pe    = info.get("trailingPE");     pe_str  = f"{pe:.2f}"    if pe    is not None else "N/A"
eps_g = info.get("earningsGrowth"); eps_str = f"{eps_g:.2%}" if eps_g is not None else "N/A"
de    = info.get("debtToEquity");   de_str  = f"{de:.2f}"    if de    is not None else "N/A"
cash  = info.get("totalCash");      c_str   = f"{cash:,.0f}" if cash  is not None else "N/A"
debt  = info.get("totalDebt");      d_str   = f"{debt:,.0f}" if debt  is not None else "N/A"
icr   = info.get("currentRatio");   icr_str = f"{icr:.2f}"   if icr   is not None else "N/A"
peg   = info.get("pegRatio");       peg_str = f"{peg:.2f}"   if peg   is not None else "N/A"
div_y = info.get("dividendYield");  div_str = f"{div_y:.2%}" if div_y is not None else "N/A"

print(f"  P/E Ratio        : {pe_str}")
print(f"  EPS Growth (TTM) : {eps_str}")
print(f"  Debt / Equity    : {de_str}")
print(f"  Total Cash       : ₹{c_str}")
print(f"  Total Debt       : ₹{d_str}")
print(f"  Current Ratio    : {icr_str}")
print(f"  PEG Ratio        : {peg_str}")
print(f"  Dividend Yield   : {div_str}")

### Observations

_TODO (AV): Write 3–5 paragraphs on what the technical and fundamental data reveals about TCS.NS. Consider: trend direction, momentum signals (RSI / MACD), valuation (P/E, PEG), and financial health (D/E, cash position)._

## Section 3 — INFY.NS | AK (IT)

In [ ]:
# ── 3a. Fetch ──────────────────────────────────────────────────────────────
ticker = TICKERS["AK"]
df     = fetch_stock(ticker)
print(f"{ticker} | {df.shape[0]} rows | {df.index[0].date()} → {df.index[-1].date()}")
df.tail()

In [ ]:
# ── 3b. Summary Statistics ──────────────────────────────────────────────────
print(f"Shape     : {df.shape}")
print(f"Date range: {df.index[0].date()} → {df.index[-1].date()}")
df.describe().round(2)

In [ ]:
# ── 3c. Price: Candlestick + EMA20 / EMA50 ────────────────────────────────
ema20 = EMAIndicator(close=df["Close"], window=20).ema_indicator()
ema50 = EMAIndicator(close=df["Close"], window=50).ema_indicator()

fig = make_subplots(rows=1, cols=1)
fig.add_trace(go.Candlestick(
    x=df.index, open=df["Open"], high=df["High"],
    low=df["Low"], close=df["Close"], name="OHLC",
))
fig.add_trace(go.Scatter(x=df.index, y=ema20, name="EMA20", line=dict(color="orange", width=1.5)))
fig.add_trace(go.Scatter(x=df.index, y=ema50, name="EMA50", line=dict(color="#4A90D9", width=1.5)))
fig.update_layout(
    title=f"{ticker} — Price + EMA20/50",
    xaxis_rangeslider_visible=False,
    height=500,
)
fig.show()

In [ ]:
# ── 3d. MACD (12/26/9) ────────────────────────────────────────────────────
macd_ind    = MACD(close=df["Close"], window_slow=26, window_fast=12, window_sign=9)
macd_line   = macd_ind.macd()
signal_line = macd_ind.macd_signal()
histogram   = macd_ind.macd_diff()

fig = make_subplots(
    rows=2, cols=1, shared_xaxes=True,
    row_heights=[0.6, 0.4],
    subplot_titles=[f"{ticker} — Close", "MACD (12/26/9)"],
)
fig.add_trace(go.Scatter(x=df.index, y=df["Close"], name="Close",  line=dict(color="white")),  row=1, col=1)
fig.add_trace(go.Scatter(x=df.index, y=macd_line,   name="MACD",   line=dict(color="cyan")),   row=2, col=1)
fig.add_trace(go.Scatter(x=df.index, y=signal_line, name="Signal", line=dict(color="orange")), row=2, col=1)
fig.add_trace(go.Bar(    x=df.index, y=histogram,   name="Hist",   marker_color="white"),       row=2, col=1)
fig.update_layout(height=600, xaxis_rangeslider_visible=False)
fig.show()

In [ ]:
# ── 3e. RSI (14) ───────────────────────────────────────────────────────────
rsi = RSIIndicator(close=df["Close"], window=14).rsi()

fig = go.Figure()
fig.add_trace(go.Scatter(x=df.index, y=rsi, name="RSI(14)", line=dict(color="mediumpurple", width=1.5)))
fig.add_hline(y=70, line_dash="dash", line_color="red",       annotation_text="Overbought (70)")
fig.add_hline(y=30, line_dash="dash", line_color="limegreen", annotation_text="Oversold (30)")
fig.update_layout(
    title=f"{ticker} — RSI (14)",
    yaxis=dict(range=[0, 100]),
    height=350,
)
fig.show()

In [ ]:
# ── 3f. Volume ──────────────────────────────────────────────────────────────
vol_avg = df["Volume"].rolling(20).mean()

fig = go.Figure()
fig.add_trace(go.Bar(   x=df.index, y=df["Volume"], name="Volume",  marker_color="white", opacity=0.6))
fig.add_trace(go.Scatter(x=df.index, y=vol_avg,     name="20d Avg", line=dict(color="orange", width=1.5)))
fig.update_layout(title=f"{ticker} — Volume + 20-day Avg", height=350)
fig.show()

In [ ]:
# ── 3g. Fundamentals ─────────────────────────────────────────────────────────
ticker = TICKERS["AK"]
info   = yf.Ticker(ticker).info

pe    = info.get("trailingPE");     pe_str  = f"{pe:.2f}"    if pe    is not None else "N/A"
eps_g = info.get("earningsGrowth"); eps_str = f"{eps_g:.2%}" if eps_g is not None else "N/A"
de    = info.get("debtToEquity");   de_str  = f"{de:.2f}"    if de    is not None else "N/A"
cash  = info.get("totalCash");      c_str   = f"{cash:,.0f}" if cash  is not None else "N/A"
debt  = info.get("totalDebt");      d_str   = f"{debt:,.0f}" if debt  is not None else "N/A"
icr   = info.get("currentRatio");   icr_str = f"{icr:.2f}"   if icr   is not None else "N/A"
peg   = info.get("pegRatio");       peg_str = f"{peg:.2f}"   if peg   is not None else "N/A"
div_y = info.get("dividendYield");  div_str = f"{div_y:.2%}" if div_y is not None else "N/A"

print(f"  P/E Ratio        : {pe_str}")
print(f"  EPS Growth (TTM) : {eps_str}")
print(f"  Debt / Equity    : {de_str}")
print(f"  Total Cash       : ₹{c_str}")
print(f"  Total Debt       : ₹{d_str}")
print(f"  Current Ratio    : {icr_str}")
print(f"  PEG Ratio        : {peg_str}")
print(f"  Dividend Yield   : {div_str}")

### Observations

_TODO (AK): Write 3–5 paragraphs on what the technical and fundamental data reveals about INFY.NS. Consider: trend direction, momentum signals (RSI / MACD), valuation (P/E, PEG), and financial health (D/E, cash position)._

## Section 4 — HCLTECH.NS | SS (IT)

In [ ]:
# ── 4a. Fetch ──────────────────────────────────────────────────────────────
ticker = TICKERS["SS"]
df     = fetch_stock(ticker)
print(f"{ticker} | {df.shape[0]} rows | {df.index[0].date()} → {df.index[-1].date()}")
df.tail()

In [ ]:
# ── 4b. Summary Statistics ──────────────────────────────────────────────────
print(f"Shape     : {df.shape}")
print(f"Date range: {df.index[0].date()} → {df.index[-1].date()}")
df.describe().round(2)

In [ ]:
# ── 4c. Price: Candlestick + EMA20 / EMA50 ────────────────────────────────
ema20 = EMAIndicator(close=df["Close"], window=20).ema_indicator()
ema50 = EMAIndicator(close=df["Close"], window=50).ema_indicator()

fig = make_subplots(rows=1, cols=1)
fig.add_trace(go.Candlestick(
    x=df.index, open=df["Open"], high=df["High"],
    low=df["Low"], close=df["Close"], name="OHLC",
))
fig.add_trace(go.Scatter(x=df.index, y=ema20, name="EMA20", line=dict(color="orange", width=1.5)))
fig.add_trace(go.Scatter(x=df.index, y=ema50, name="EMA50", line=dict(color="#4A90D9", width=1.5)))
fig.update_layout(
    title=f"{ticker} — Price + EMA20/50",
    xaxis_rangeslider_visible=False,
    height=500,
)
fig.show()

In [ ]:
# ── 4d. MACD (12/26/9) ────────────────────────────────────────────────────
macd_ind    = MACD(close=df["Close"], window_slow=26, window_fast=12, window_sign=9)
macd_line   = macd_ind.macd()
signal_line = macd_ind.macd_signal()
histogram   = macd_ind.macd_diff()

fig = make_subplots(
    rows=2, cols=1, shared_xaxes=True,
    row_heights=[0.6, 0.4],
    subplot_titles=[f"{ticker} — Close", "MACD (12/26/9)"],
)
fig.add_trace(go.Scatter(x=df.index, y=df["Close"], name="Close",  line=dict(color="white")),  row=1, col=1)
fig.add_trace(go.Scatter(x=df.index, y=macd_line,   name="MACD",   line=dict(color="cyan")),   row=2, col=1)
fig.add_trace(go.Scatter(x=df.index, y=signal_line, name="Signal", line=dict(color="orange")), row=2, col=1)
fig.add_trace(go.Bar(    x=df.index, y=histogram,   name="Hist",   marker_color="white"),       row=2, col=1)
fig.update_layout(height=600, xaxis_rangeslider_visible=False)
fig.show()

In [ ]:
# ── 4e. RSI (14) ───────────────────────────────────────────────────────────
rsi = RSIIndicator(close=df["Close"], window=14).rsi()

fig = go.Figure()
fig.add_trace(go.Scatter(x=df.index, y=rsi, name="RSI(14)", line=dict(color="mediumpurple", width=1.5)))
fig.add_hline(y=70, line_dash="dash", line_color="red",       annotation_text="Overbought (70)")
fig.add_hline(y=30, line_dash="dash", line_color="limegreen", annotation_text="Oversold (30)")
fig.update_layout(
    title=f"{ticker} — RSI (14)",
    yaxis=dict(range=[0, 100]),
    height=350,
)
fig.show()

In [ ]:
# ── 4f. Volume ──────────────────────────────────────────────────────────────
vol_avg = df["Volume"].rolling(20).mean()

fig = go.Figure()
fig.add_trace(go.Bar(   x=df.index, y=df["Volume"], name="Volume",  marker_color="white", opacity=0.6))
fig.add_trace(go.Scatter(x=df.index, y=vol_avg,     name="20d Avg", line=dict(color="orange", width=1.5)))
fig.update_layout(title=f"{ticker} — Volume + 20-day Avg", height=350)
fig.show()

In [ ]:
# ── 4g. Fundamentals ─────────────────────────────────────────────────────────
ticker = TICKERS["SS"]
info   = yf.Ticker(ticker).info

pe    = info.get("trailingPE");     pe_str  = f"{pe:.2f}"    if pe    is not None else "N/A"
eps_g = info.get("earningsGrowth"); eps_str = f"{eps_g:.2%}" if eps_g is not None else "N/A"
de    = info.get("debtToEquity");   de_str  = f"{de:.2f}"    if de    is not None else "N/A"
cash  = info.get("totalCash");      c_str   = f"{cash:,.0f}" if cash  is not None else "N/A"
debt  = info.get("totalDebt");      d_str   = f"{debt:,.0f}" if debt  is not None else "N/A"
icr   = info.get("currentRatio");   icr_str = f"{icr:.2f}"   if icr   is not None else "N/A"
peg   = info.get("pegRatio");       peg_str = f"{peg:.2f}"   if peg   is not None else "N/A"
div_y = info.get("dividendYield");  div_str = f"{div_y:.2%}" if div_y is not None else "N/A"

print(f"  P/E Ratio        : {pe_str}")
print(f"  EPS Growth (TTM) : {eps_str}")
print(f"  Debt / Equity    : {de_str}")
print(f"  Total Cash       : ₹{c_str}")
print(f"  Total Debt       : ₹{d_str}")
print(f"  Current Ratio    : {icr_str}")
print(f"  PEG Ratio        : {peg_str}")
print(f"  Dividend Yield   : {div_str}")

### Observations

_TODO (SS): Write 3–5 paragraphs on what the technical and fundamental data reveals about HCLTECH.NS. Consider: trend direction, momentum signals (RSI / MACD), valuation (P/E, PEG), and financial health (D/E, cash position)._

## Banking

## Section 5 — HDFCBANK.NS | AR (Banking)

In [4]:
# ── 5a. Fetch ──────────────────────────────────────────────────────────────
ticker = TICKERS["AR"]
df     = fetch_stock(ticker)
print(f"{ticker} | {df.shape[0]} rows | {df.index[0].date()} → {df.index[-1].date()}")
df.tail()

HDFCBANK.NS | 250 rows | 2025-06-05 → 2026-06-05


Price,Open,High,Low,Close,Volume
Date,,,,,
2026-06-01,749.000000,752.349976,739.200012,742.700012,47996441
2026-06-02,737.000000,753.599976,733.150024,748.250000,47314447
2026-06-03,744.450012,756.900024,742.599976,753.650024,36109480
2026-06-04,749.150024,757.299988,745.000000,754.200012,42673538
2026-06-05,753.950012,758.700012,744.650024,747.049988,22116568


In [5]:
# ── 5b. Summary Statistics ──────────────────────────────────────────────────
print(f"Shape     : {df.shape}")
print(f"Date range: {df.index[0].date()} → {df.index[-1].date()}")
df.describe().round(2)

Shape     : (250, 5)
Date range: 2025-06-05 → 2026-06-05


Price,Open,High,Low,Close,Volume
count,250.00,250.00,250.00,250.00,2.500000e+02
mean,924.63,932.37,918.10,924.97,2.705422e+07
std,87.80,86.35,88.14,87.17,1.861607e+07
min,730.00,751.00,726.65,731.55,0.000000e+00
25%,869.50,879.82,857.20,870.93,1.517161e+07
50%,962.35,969.47,956.71,963.80,2.218912e+07
75%,991.23,998.51,985.95,992.00,3.457548e+07
max,1017.50,1020.50,1008.50,1012.90,1.716374e+08


In [6]:
# ── 5c. Price: Candlestick + EMA20 / EMA50 ────────────────────────────────
ema20 = EMAIndicator(close=df["Close"], window=20).ema_indicator()
ema50 = EMAIndicator(close=df["Close"], window=50).ema_indicator()

fig = make_subplots(rows=1, cols=1)
fig.add_trace(go.Candlestick(
    x=df.index, open=df["Open"], high=df["High"],
    low=df["Low"], close=df["Close"], name="OHLC",
))
fig.add_trace(go.Scatter(x=df.index, y=ema20, name="EMA20", line=dict(color="orange", width=1.5)))
fig.add_trace(go.Scatter(x=df.index, y=ema50, name="EMA50", line=dict(color="#4A90D9", width=1.5)))
fig.update_layout(
    title=f"{ticker} — Price + EMA20/50",
    xaxis_rangeslider_visible=False,
    height=500,
)
fig.show()

In [7]:
# ── 5d. MACD (12/26/9) ────────────────────────────────────────────────────
macd_ind    = MACD(close=df["Close"], window_slow=26, window_fast=12, window_sign=9)
macd_line   = macd_ind.macd()
signal_line = macd_ind.macd_signal()
histogram   = macd_ind.macd_diff()

fig = make_subplots(
    rows=2, cols=1, shared_xaxes=True,
    row_heights=[0.6, 0.4],
    subplot_titles=[f"{ticker} — Close", "MACD (12/26/9)"],
)
fig.add_trace(go.Scatter(x=df.index, y=df["Close"], name="Close",  line=dict(color="white")),  row=1, col=1)
fig.add_trace(go.Scatter(x=df.index, y=macd_line,   name="MACD",   line=dict(color="cyan")),   row=2, col=1)
fig.add_trace(go.Scatter(x=df.index, y=signal_line, name="Signal", line=dict(color="orange")), row=2, col=1)
fig.add_trace(go.Bar(    x=df.index, y=histogram,   name="Hist",   marker_color="white"),       row=2, col=1)
fig.update_layout(height=600, xaxis_rangeslider_visible=False)
fig.show()

In [8]:
# ── 5e. RSI (14) ───────────────────────────────────────────────────────────
rsi = RSIIndicator(close=df["Close"], window=14).rsi()

fig = go.Figure()
fig.add_trace(go.Scatter(x=df.index, y=rsi, name="RSI(14)", line=dict(color="mediumpurple", width=1.5)))
fig.add_hline(y=70, line_dash="dash", line_color="red",       annotation_text="Overbought (70)")
fig.add_hline(y=30, line_dash="dash", line_color="limegreen", annotation_text="Oversold (30)")
fig.update_layout(
    title=f"{ticker} — RSI (14)",
    yaxis=dict(range=[0, 100]),
    height=350,
)
fig.show()

In [9]:
# ── 5f. Volume ──────────────────────────────────────────────────────────────
vol_avg = df["Volume"].rolling(20).mean()

fig = go.Figure()
fig.add_trace(go.Bar(   x=df.index, y=df["Volume"], name="Volume",  marker_color="white", opacity=0.6))
fig.add_trace(go.Scatter(x=df.index, y=vol_avg,     name="20d Avg", line=dict(color="orange", width=1.5)))
fig.update_layout(title=f"{ticker} — Volume + 20-day Avg", height=350)
fig.show()

In [10]:
# ── 5g. Fundamentals ─────────────────────────────────────────────────────────
ticker = TICKERS["AR"]
info   = yf.Ticker(ticker).info

pe    = info.get("trailingPE");     pe_str  = f"{pe:.2f}"    if pe    is not None else "N/A"
eps_g = info.get("earningsGrowth"); eps_str = f"{eps_g:.2%}" if eps_g is not None else "N/A"
de    = info.get("debtToEquity");   de_str  = f"{de:.2f}"    if de    is not None else "N/A"
cash  = info.get("totalCash");      c_str   = f"{cash:,.0f}" if cash  is not None else "N/A"
debt  = info.get("totalDebt");      d_str   = f"{debt:,.0f}" if debt  is not None else "N/A"
icr   = info.get("currentRatio");   icr_str = f"{icr:.2f}"   if icr   is not None else "N/A"
peg   = info.get("pegRatio");       peg_str = f"{peg:.2f}"   if peg   is not None else "N/A"
div_y = info.get("dividendYield");  div_str = f"{div_y:.2%}" if div_y is not None else "N/A"

print(f"  P/E Ratio        : {pe_str}")
print(f"  EPS Growth (TTM) : {eps_str}")
print(f"  Debt / Equity    : {de_str}")
print(f"  Total Cash       : ₹{c_str}")
print(f"  Total Debt       : ₹{d_str}")
print(f"  Current Ratio    : {icr_str}")
print(f"  PEG Ratio        : {peg_str}")
print(f"  Dividend Yield   : {div_str}")

  P/E Ratio        : 16.67
  EPS Growth (TTM) : 7.50%
  Debt / Equity    : N/A
  Total Cash       : ₹3,119,260,631,040
  Total Debt       : ₹5,884,845,490,176
  Current Ratio    : N/A
  PEG Ratio        : 0.89
  Dividend Yield   : 174.00%


### Observations
Based on the charts and statistics, HDFCBANK.NS has shown a generally downward trend during the last one year. The stock price was trading near ₹1,000 in the middle of 2025 but gradually declined and is currently trading around ₹747. Both the 20-day and 50-day Exponential Moving Averages (EMA20 and EMA50) are above the current price, which indicates that the stock has been under selling pressure in recent months.

The MACD chart also reflects weak momentum. Although the MACD and Signal lines have moved closer together recently, they remain below the zero line, suggesting that bearish sentiment is still present. However, the gap between the lines has narrowed compared to earlier months, which may indicate that the downward momentum is slowing.

The Relative Strength Index (RSI) is currently around the middle range and is neither in the overbought zone (above 70) nor the oversold zone (below 30). This suggests that the stock is not experiencing extreme buying or selling activity at the moment. The RSI recovered from oversold levels earlier in the year, indicating some improvement in market sentiment.

From a volume perspective, trading activity increased significantly during March–April 2026, as seen from the sharp spikes in volume. Higher volume often indicates increased investor interest. Recently, volume has moved closer to its 20-day average, suggesting more stable trading activity.

Looking at the fundamental indicators, the company appears reasonably valued with a P/E ratio of about 16.67 and a PEG ratio of 0.89. The company also maintains a strong cash position of over ₹3.1 trillion, although total debt is also substantial. Overall, HDFC Bank appears financially stable, but the technical indicators suggest that the stock is still recovering from a prolonged decline. As a college student reviewing this data, I would conclude that HDFC Bank remains a fundamentally strong company, but investors may wait for clearer signs of an upward trend before expecting significant price appreciation.

## Section 6 — ICICIBANK.NS | EB (Banking)

In [ ]:
# ── 6a. Fetch ──────────────────────────────────────────────────────────────
ticker = TICKERS["EB"]
df     = fetch_stock(ticker)
print(f"{ticker} | {df.shape[0]} rows | {df.index[0].date()} → {df.index[-1].date()}")
df.tail()

In [ ]:
# ── 6b. Summary Statistics ──────────────────────────────────────────────────
print(f"Shape     : {df.shape}")
print(f"Date range: {df.index[0].date()} → {df.index[-1].date()}")
df.describe().round(2)

In [ ]:
# ── 6c. Price: Candlestick + EMA20 / EMA50 ────────────────────────────────
ema20 = EMAIndicator(close=df["Close"], window=20).ema_indicator()
ema50 = EMAIndicator(close=df["Close"], window=50).ema_indicator()

fig = make_subplots(rows=1, cols=1)
fig.add_trace(go.Candlestick(
    x=df.index, open=df["Open"], high=df["High"],
    low=df["Low"], close=df["Close"], name="OHLC",
))
fig.add_trace(go.Scatter(x=df.index, y=ema20, name="EMA20", line=dict(color="orange", width=1.5)))
fig.add_trace(go.Scatter(x=df.index, y=ema50, name="EMA50", line=dict(color="#4A90D9", width=1.5)))
fig.update_layout(
    title=f"{ticker} — Price + EMA20/50",
    xaxis_rangeslider_visible=False,
    height=500,
)
fig.show()

In [ ]:
# ── 6d. MACD (12/26/9) ────────────────────────────────────────────────────
macd_ind    = MACD(close=df["Close"], window_slow=26, window_fast=12, window_sign=9)
macd_line   = macd_ind.macd()
signal_line = macd_ind.macd_signal()
histogram   = macd_ind.macd_diff()

fig = make_subplots(
    rows=2, cols=1, shared_xaxes=True,
    row_heights=[0.6, 0.4],
    subplot_titles=[f"{ticker} — Close", "MACD (12/26/9)"],
)
fig.add_trace(go.Scatter(x=df.index, y=df["Close"], name="Close",  line=dict(color="white")),  row=1, col=1)
fig.add_trace(go.Scatter(x=df.index, y=macd_line,   name="MACD",   line=dict(color="cyan")),   row=2, col=1)
fig.add_trace(go.Scatter(x=df.index, y=signal_line, name="Signal", line=dict(color="orange")), row=2, col=1)
fig.add_trace(go.Bar(    x=df.index, y=histogram,   name="Hist",   marker_color="white"),       row=2, col=1)
fig.update_layout(height=600, xaxis_rangeslider_visible=False)
fig.show()

In [ ]:
# ── 6e. RSI (14) ───────────────────────────────────────────────────────────
rsi = RSIIndicator(close=df["Close"], window=14).rsi()

fig = go.Figure()
fig.add_trace(go.Scatter(x=df.index, y=rsi, name="RSI(14)", line=dict(color="mediumpurple", width=1.5)))
fig.add_hline(y=70, line_dash="dash", line_color="red",       annotation_text="Overbought (70)")
fig.add_hline(y=30, line_dash="dash", line_color="limegreen", annotation_text="Oversold (30)")
fig.update_layout(
    title=f"{ticker} — RSI (14)",
    yaxis=dict(range=[0, 100]),
    height=350,
)
fig.show()

In [ ]:
# ── 6f. Volume ──────────────────────────────────────────────────────────────
vol_avg = df["Volume"].rolling(20).mean()

fig = go.Figure()
fig.add_trace(go.Bar(   x=df.index, y=df["Volume"], name="Volume",  marker_color="white", opacity=0.6))
fig.add_trace(go.Scatter(x=df.index, y=vol_avg,     name="20d Avg", line=dict(color="orange", width=1.5)))
fig.update_layout(title=f"{ticker} — Volume + 20-day Avg", height=350)
fig.show()

In [ ]:
# ── 6g. Fundamentals ─────────────────────────────────────────────────────────
ticker = TICKERS["EB"]
info   = yf.Ticker(ticker).info

pe    = info.get("trailingPE");     pe_str  = f"{pe:.2f}"    if pe    is not None else "N/A"
eps_g = info.get("earningsGrowth"); eps_str = f"{eps_g:.2%}" if eps_g is not None else "N/A"
de    = info.get("debtToEquity");   de_str  = f"{de:.2f}"    if de    is not None else "N/A"
cash  = info.get("totalCash");      c_str   = f"{cash:,.0f}" if cash  is not None else "N/A"
debt  = info.get("totalDebt");      d_str   = f"{debt:,.0f}" if debt  is not None else "N/A"
icr   = info.get("currentRatio");   icr_str = f"{icr:.2f}"   if icr   is not None else "N/A"
peg   = info.get("pegRatio");       peg_str = f"{peg:.2f}"   if peg   is not None else "N/A"
div_y = info.get("dividendYield");  div_str = f"{div_y:.2%}" if div_y is not None else "N/A"

print(f"  P/E Ratio        : {pe_str}")
print(f"  EPS Growth (TTM) : {eps_str}")
print(f"  Debt / Equity    : {de_str}")
print(f"  Total Cash       : ₹{c_str}")
print(f"  Total Debt       : ₹{d_str}")
print(f"  Current Ratio    : {icr_str}")
print(f"  PEG Ratio        : {peg_str}")
print(f"  Dividend Yield   : {div_str}")

### Observations

_TODO (EB): Write 3–5 paragraphs on what the technical and fundamental data reveals about ICICIBANK.NS. Consider: trend direction, momentum signals (RSI / MACD), valuation (P/E, PEG), and financial health (D/E, cash position)._

## Section 7 — AXISBANK.NS | ShS (Banking)

In [ ]:
# ── 7a. Fetch ──────────────────────────────────────────────────────────────
ticker = TICKERS["ShS"]
df     = fetch_stock(ticker)
print(f"{ticker} | {df.shape[0]} rows | {df.index[0].date()} → {df.index[-1].date()}")
df.tail()

In [ ]:
# ── 7b. Summary Statistics ──────────────────────────────────────────────────
print(f"Shape     : {df.shape}")
print(f"Date range: {df.index[0].date()} → {df.index[-1].date()}")
df.describe().round(2)

In [ ]:
# ── 7c. Price: Candlestick + EMA20 / EMA50 ────────────────────────────────
ema20 = EMAIndicator(close=df["Close"], window=20).ema_indicator()
ema50 = EMAIndicator(close=df["Close"], window=50).ema_indicator()

fig = make_subplots(rows=1, cols=1)
fig.add_trace(go.Candlestick(
    x=df.index, open=df["Open"], high=df["High"],
    low=df["Low"], close=df["Close"], name="OHLC",
))
fig.add_trace(go.Scatter(x=df.index, y=ema20, name="EMA20", line=dict(color="orange", width=1.5)))
fig.add_trace(go.Scatter(x=df.index, y=ema50, name="EMA50", line=dict(color="#4A90D9", width=1.5)))
fig.update_layout(
    title=f"{ticker} — Price + EMA20/50",
    xaxis_rangeslider_visible=False,
    height=500,
)
fig.show()

In [ ]:
# ── 7d. MACD (12/26/9) ────────────────────────────────────────────────────
macd_ind    = MACD(close=df["Close"], window_slow=26, window_fast=12, window_sign=9)
macd_line   = macd_ind.macd()
signal_line = macd_ind.macd_signal()
histogram   = macd_ind.macd_diff()

fig = make_subplots(
    rows=2, cols=1, shared_xaxes=True,
    row_heights=[0.6, 0.4],
    subplot_titles=[f"{ticker} — Close", "MACD (12/26/9)"],
)
fig.add_trace(go.Scatter(x=df.index, y=df["Close"], name="Close",  line=dict(color="white")),  row=1, col=1)
fig.add_trace(go.Scatter(x=df.index, y=macd_line,   name="MACD",   line=dict(color="cyan")),   row=2, col=1)
fig.add_trace(go.Scatter(x=df.index, y=signal_line, name="Signal", line=dict(color="orange")), row=2, col=1)
fig.add_trace(go.Bar(    x=df.index, y=histogram,   name="Hist",   marker_color="white"),       row=2, col=1)
fig.update_layout(height=600, xaxis_rangeslider_visible=False)
fig.show()

In [ ]:
# ── 7e. RSI (14) ───────────────────────────────────────────────────────────
rsi = RSIIndicator(close=df["Close"], window=14).rsi()

fig = go.Figure()
fig.add_trace(go.Scatter(x=df.index, y=rsi, name="RSI(14)", line=dict(color="mediumpurple", width=1.5)))
fig.add_hline(y=70, line_dash="dash", line_color="red",       annotation_text="Overbought (70)")
fig.add_hline(y=30, line_dash="dash", line_color="limegreen", annotation_text="Oversold (30)")
fig.update_layout(
    title=f"{ticker} — RSI (14)",
    yaxis=dict(range=[0, 100]),
    height=350,
)
fig.show()

In [ ]:
# ── 7f. Volume ──────────────────────────────────────────────────────────────
vol_avg = df["Volume"].rolling(20).mean()

fig = go.Figure()
fig.add_trace(go.Bar(   x=df.index, y=df["Volume"], name="Volume",  marker_color="white", opacity=0.6))
fig.add_trace(go.Scatter(x=df.index, y=vol_avg,     name="20d Avg", line=dict(color="orange", width=1.5)))
fig.update_layout(title=f"{ticker} — Volume + 20-day Avg", height=350)
fig.show()

In [ ]:
# ── 7g. Fundamentals ─────────────────────────────────────────────────────────
ticker = TICKERS["ShS"]
info   = yf.Ticker(ticker).info

pe    = info.get("trailingPE");     pe_str  = f"{pe:.2f}"    if pe    is not None else "N/A"
eps_g = info.get("earningsGrowth"); eps_str = f"{eps_g:.2%}" if eps_g is not None else "N/A"
de    = info.get("debtToEquity");   de_str  = f"{de:.2f}"    if de    is not None else "N/A"
cash  = info.get("totalCash");      c_str   = f"{cash:,.0f}" if cash  is not None else "N/A"
debt  = info.get("totalDebt");      d_str   = f"{debt:,.0f}" if debt  is not None else "N/A"
icr   = info.get("currentRatio");   icr_str = f"{icr:.2f}"   if icr   is not None else "N/A"
peg   = info.get("pegRatio");       peg_str = f"{peg:.2f}"   if peg   is not None else "N/A"
div_y = info.get("dividendYield");  div_str = f"{div_y:.2%}" if div_y is not None else "N/A"

print(f"  P/E Ratio        : {pe_str}")
print(f"  EPS Growth (TTM) : {eps_str}")
print(f"  Debt / Equity    : {de_str}")
print(f"  Total Cash       : ₹{c_str}")
print(f"  Total Debt       : ₹{d_str}")
print(f"  Current Ratio    : {icr_str}")
print(f"  PEG Ratio        : {peg_str}")
print(f"  Dividend Yield   : {div_str}")

### Observations

_TODO (ShS): Write 3–5 paragraphs on what the technical and fundamental data reveals about AXISBANK.NS. Consider: trend direction, momentum signals (RSI / MACD), valuation (P/E, PEG), and financial health (D/E, cash position)._

## Auto

## Section 8 — TVSMOTOR.NS | RS (Auto)

In [ ]:
# ── 8a. Fetch ──────────────────────────────────────────────────────────────
ticker = TICKERS["RS"]
df     = fetch_stock(ticker)
print(f"{ticker} | {df.shape[0]} rows | {df.index[0].date()} → {df.index[-1].date()}")
df.tail()

In [ ]:
# ── 8b. Summary Statistics ──────────────────────────────────────────────────
print(f"Shape     : {df.shape}")
print(f"Date range: {df.index[0].date()} → {df.index[-1].date()}")
df.describe().round(2)

In [ ]:
# ── 8c. Price: Candlestick + EMA20 / EMA50 ────────────────────────────────
ema20 = EMAIndicator(close=df["Close"], window=20).ema_indicator()
ema50 = EMAIndicator(close=df["Close"], window=50).ema_indicator()

fig = make_subplots(rows=1, cols=1)
fig.add_trace(go.Candlestick(
    x=df.index, open=df["Open"], high=df["High"],
    low=df["Low"], close=df["Close"], name="OHLC",
))
fig.add_trace(go.Scatter(x=df.index, y=ema20, name="EMA20", line=dict(color="orange", width=1.5)))
fig.add_trace(go.Scatter(x=df.index, y=ema50, name="EMA50", line=dict(color="#4A90D9", width=1.5)))
fig.update_layout(
    title=f"{ticker} — Price + EMA20/50",
    xaxis_rangeslider_visible=False,
    height=500,
)
fig.show()

In [ ]:
# ── 8d. MACD (12/26/9) ────────────────────────────────────────────────────
macd_ind    = MACD(close=df["Close"], window_slow=26, window_fast=12, window_sign=9)
macd_line   = macd_ind.macd()
signal_line = macd_ind.macd_signal()
histogram   = macd_ind.macd_diff()

fig = make_subplots(
    rows=2, cols=1, shared_xaxes=True,
    row_heights=[0.6, 0.4],
    subplot_titles=[f"{ticker} — Close", "MACD (12/26/9)"],
)
fig.add_trace(go.Scatter(x=df.index, y=df["Close"], name="Close",  line=dict(color="white")),  row=1, col=1)
fig.add_trace(go.Scatter(x=df.index, y=macd_line,   name="MACD",   line=dict(color="cyan")),   row=2, col=1)
fig.add_trace(go.Scatter(x=df.index, y=signal_line, name="Signal", line=dict(color="orange")), row=2, col=1)
fig.add_trace(go.Bar(    x=df.index, y=histogram,   name="Hist",   marker_color="white"),       row=2, col=1)
fig.update_layout(height=600, xaxis_rangeslider_visible=False)
fig.show()

In [ ]:
# ── 8e. RSI (14) ───────────────────────────────────────────────────────────
rsi = RSIIndicator(close=df["Close"], window=14).rsi()

fig = go.Figure()
fig.add_trace(go.Scatter(x=df.index, y=rsi, name="RSI(14)", line=dict(color="mediumpurple", width=1.5)))
fig.add_hline(y=70, line_dash="dash", line_color="red",       annotation_text="Overbought (70)")
fig.add_hline(y=30, line_dash="dash", line_color="limegreen", annotation_text="Oversold (30)")
fig.update_layout(
    title=f"{ticker} — RSI (14)",
    yaxis=dict(range=[0, 100]),
    height=350,
)
fig.show()

In [ ]:
# ── 8f. Volume ──────────────────────────────────────────────────────────────
vol_avg = df["Volume"].rolling(20).mean()

fig = go.Figure()
fig.add_trace(go.Bar(   x=df.index, y=df["Volume"], name="Volume",  marker_color="white", opacity=0.6))
fig.add_trace(go.Scatter(x=df.index, y=vol_avg,     name="20d Avg", line=dict(color="orange", width=1.5)))
fig.update_layout(title=f"{ticker} — Volume + 20-day Avg", height=350)
fig.show()

In [ ]:
# ── 8g. Fundamentals ─────────────────────────────────────────────────────────
ticker = TICKERS["RS"]
info   = yf.Ticker(ticker).info

pe    = info.get("trailingPE");     pe_str  = f"{pe:.2f}"    if pe    is not None else "N/A"
eps_g = info.get("earningsGrowth"); eps_str = f"{eps_g:.2%}" if eps_g is not None else "N/A"
de    = info.get("debtToEquity");   de_str  = f"{de:.2f}"    if de    is not None else "N/A"
cash  = info.get("totalCash");      c_str   = f"{cash:,.0f}" if cash  is not None else "N/A"
debt  = info.get("totalDebt");      d_str   = f"{debt:,.0f}" if debt  is not None else "N/A"
icr   = info.get("currentRatio");   icr_str = f"{icr:.2f}"   if icr   is not None else "N/A"
peg   = info.get("pegRatio");       peg_str = f"{peg:.2f}"   if peg   is not None else "N/A"
div_y = info.get("dividendYield");  div_str = f"{div_y:.2%}" if div_y is not None else "N/A"

print(f"  P/E Ratio        : {pe_str}")
print(f"  EPS Growth (TTM) : {eps_str}")
print(f"  Debt / Equity    : {de_str}")
print(f"  Total Cash       : ₹{c_str}")
print(f"  Total Debt       : ₹{d_str}")
print(f"  Current Ratio    : {icr_str}")
print(f"  PEG Ratio        : {peg_str}")
print(f"  Dividend Yield   : {div_str}")

### Observations

_TODO (RS): Write 3–5 paragraphs on what the technical and fundamental data reveals about TVSMOTOR.NS. Consider: trend direction, momentum signals (RSI / MACD), valuation (P/E, PEG), and financial health (D/E, cash position)._

## Section 9 — M&M.NS | GT (Auto)

In [ ]:
# ── 9a. Fetch ──────────────────────────────────────────────────────────────
ticker = TICKERS["GT"]
df     = fetch_stock(ticker)
print(f"{ticker} | {df.shape[0]} rows | {df.index[0].date()} → {df.index[-1].date()}")
df.tail()

In [ ]:
# ── 9b. Summary Statistics ──────────────────────────────────────────────────
print(f"Shape     : {df.shape}")
print(f"Date range: {df.index[0].date()} → {df.index[-1].date()}")
df.describe().round(2)

In [ ]:
# ── 9c. Price: Candlestick + EMA20 / EMA50 ────────────────────────────────
ema20 = EMAIndicator(close=df["Close"], window=20).ema_indicator()
ema50 = EMAIndicator(close=df["Close"], window=50).ema_indicator()

fig = make_subplots(rows=1, cols=1)
fig.add_trace(go.Candlestick(
    x=df.index, open=df["Open"], high=df["High"],
    low=df["Low"], close=df["Close"], name="OHLC",
))
fig.add_trace(go.Scatter(x=df.index, y=ema20, name="EMA20", line=dict(color="orange", width=1.5)))
fig.add_trace(go.Scatter(x=df.index, y=ema50, name="EMA50", line=dict(color="#4A90D9", width=1.5)))
fig.update_layout(
    title=f"{ticker} — Price + EMA20/50",
    xaxis_rangeslider_visible=False,
    height=500,
)
fig.show()

In [ ]:
# ── 9d. MACD (12/26/9) ────────────────────────────────────────────────────
macd_ind    = MACD(close=df["Close"], window_slow=26, window_fast=12, window_sign=9)
macd_line   = macd_ind.macd()
signal_line = macd_ind.macd_signal()
histogram   = macd_ind.macd_diff()

fig = make_subplots(
    rows=2, cols=1, shared_xaxes=True,
    row_heights=[0.6, 0.4],
    subplot_titles=[f"{ticker} — Close", "MACD (12/26/9)"],
)
fig.add_trace(go.Scatter(x=df.index, y=df["Close"], name="Close",  line=dict(color="white")),  row=1, col=1)
fig.add_trace(go.Scatter(x=df.index, y=macd_line,   name="MACD",   line=dict(color="cyan")),   row=2, col=1)
fig.add_trace(go.Scatter(x=df.index, y=signal_line, name="Signal", line=dict(color="orange")), row=2, col=1)
fig.add_trace(go.Bar(    x=df.index, y=histogram,   name="Hist",   marker_color="white"),       row=2, col=1)
fig.update_layout(height=600, xaxis_rangeslider_visible=False)
fig.show()

In [ ]:
# ── 9e. RSI (14) ───────────────────────────────────────────────────────────
rsi = RSIIndicator(close=df["Close"], window=14).rsi()

fig = go.Figure()
fig.add_trace(go.Scatter(x=df.index, y=rsi, name="RSI(14)", line=dict(color="mediumpurple", width=1.5)))
fig.add_hline(y=70, line_dash="dash", line_color="red",       annotation_text="Overbought (70)")
fig.add_hline(y=30, line_dash="dash", line_color="limegreen", annotation_text="Oversold (30)")
fig.update_layout(
    title=f"{ticker} — RSI (14)",
    yaxis=dict(range=[0, 100]),
    height=350,
)
fig.show()

In [ ]:
# ── 9f. Volume ──────────────────────────────────────────────────────────────
vol_avg = df["Volume"].rolling(20).mean()

fig = go.Figure()
fig.add_trace(go.Bar(   x=df.index, y=df["Volume"], name="Volume",  marker_color="white", opacity=0.6))
fig.add_trace(go.Scatter(x=df.index, y=vol_avg,     name="20d Avg", line=dict(color="orange", width=1.5)))
fig.update_layout(title=f"{ticker} — Volume + 20-day Avg", height=350)
fig.show()

In [ ]:
# ── 9g. Fundamentals ─────────────────────────────────────────────────────────
ticker = TICKERS["GT"]
info   = yf.Ticker(ticker).info

pe    = info.get("trailingPE");     pe_str  = f"{pe:.2f}"    if pe    is not None else "N/A"
eps_g = info.get("earningsGrowth"); eps_str = f"{eps_g:.2%}" if eps_g is not None else "N/A"
de    = info.get("debtToEquity");   de_str  = f"{de:.2f}"    if de    is not None else "N/A"
cash  = info.get("totalCash");      c_str   = f"{cash:,.0f}" if cash  is not None else "N/A"
debt  = info.get("totalDebt");      d_str   = f"{debt:,.0f}" if debt  is not None else "N/A"
icr   = info.get("currentRatio");   icr_str = f"{icr:.2f}"   if icr   is not None else "N/A"
peg   = info.get("pegRatio");       peg_str = f"{peg:.2f}"   if peg   is not None else "N/A"
div_y = info.get("dividendYield");  div_str = f"{div_y:.2%}" if div_y is not None else "N/A"

print(f"  P/E Ratio        : {pe_str}")
print(f"  EPS Growth (TTM) : {eps_str}")
print(f"  Debt / Equity    : {de_str}")
print(f"  Total Cash       : ₹{c_str}")
print(f"  Total Debt       : ₹{d_str}")
print(f"  Current Ratio    : {icr_str}")
print(f"  PEG Ratio        : {peg_str}")
print(f"  Dividend Yield   : {div_str}")

### Section 9 — M&M.NS | GT (Auto)

####  Strategic & Technical Overviews
* **Moving Average Alignment:** The stock exhibits a strong technical setup as M&M.NS trades cleanly above both its computed EMA20 and EMA50 moving price baselines.
* **Momentum Profiling:** The MACD histogram direction confirms steady bullish momentum in the short term. Additionally, the 14-period RSI is securely positioned in the active zone, reflecting sustained buying interest without entering overbought territory.

####  Fundamental & Quantitative Summary
* **Valuation Framework:** Fundamentally, the position is well-supported by healthy P/E and forward PEG ratios tracked directly from the yfinance data outputs.
* **Balance Sheet Health:** Retains strong institutional inflow buffers backing its localized EV expansion targets.

## Infrastructure

## Section 10 — LT.NS | RT (Infrastructure)

In [ ]:
# ── 10a. Fetch ──────────────────────────────────────────────────────────────
ticker = TICKERS["RT"]
df     = fetch_stock(ticker)
print(f"{ticker} | {df.shape[0]} rows | {df.index[0].date()} → {df.index[-1].date()}")
df.tail()

In [ ]:
# ── 10b. Summary Statistics ──────────────────────────────────────────────────
print(f"Shape     : {df.shape}")
print(f"Date range: {df.index[0].date()} → {df.index[-1].date()}")
df.describe().round(2)

In [ ]:
# ── 10c. Price: Candlestick + EMA20 / EMA50 ────────────────────────────────
ema20 = EMAIndicator(close=df["Close"], window=20).ema_indicator()
ema50 = EMAIndicator(close=df["Close"], window=50).ema_indicator()

fig = make_subplots(rows=1, cols=1)
fig.add_trace(go.Candlestick(
    x=df.index, open=df["Open"], high=df["High"],
    low=df["Low"], close=df["Close"], name="OHLC",
))
fig.add_trace(go.Scatter(x=df.index, y=ema20, name="EMA20", line=dict(color="orange", width=1.5)))
fig.add_trace(go.Scatter(x=df.index, y=ema50, name="EMA50", line=dict(color="#4A90D9", width=1.5)))
fig.update_layout(
    title=f"{ticker} — Price + EMA20/50",
    xaxis_rangeslider_visible=False,
    height=500,
)
fig.show()

In [ ]:
# ── 10d. MACD (12/26/9) ────────────────────────────────────────────────────
macd_ind    = MACD(close=df["Close"], window_slow=26, window_fast=12, window_sign=9)
macd_line   = macd_ind.macd()
signal_line = macd_ind.macd_signal()
histogram   = macd_ind.macd_diff()

fig = make_subplots(
    rows=2, cols=1, shared_xaxes=True,
    row_heights=[0.6, 0.4],
    subplot_titles=[f"{ticker} — Close", "MACD (12/26/9)"],
)
fig.add_trace(go.Scatter(x=df.index, y=df["Close"], name="Close",  line=dict(color="white")),  row=1, col=1)
fig.add_trace(go.Scatter(x=df.index, y=macd_line,   name="MACD",   line=dict(color="cyan")),   row=2, col=1)
fig.add_trace(go.Scatter(x=df.index, y=signal_line, name="Signal", line=dict(color="orange")), row=2, col=1)
fig.add_trace(go.Bar(    x=df.index, y=histogram,   name="Hist",   marker_color="white"),       row=2, col=1)
fig.update_layout(height=600, xaxis_rangeslider_visible=False)
fig.show()

In [ ]:
# ── 10e. RSI (14) ───────────────────────────────────────────────────────────
rsi = RSIIndicator(close=df["Close"], window=14).rsi()

fig = go.Figure()
fig.add_trace(go.Scatter(x=df.index, y=rsi, name="RSI(14)", line=dict(color="mediumpurple", width=1.5)))
fig.add_hline(y=70, line_dash="dash", line_color="red",       annotation_text="Overbought (70)")
fig.add_hline(y=30, line_dash="dash", line_color="limegreen", annotation_text="Oversold (30)")
fig.update_layout(
    title=f"{ticker} — RSI (14)",
    yaxis=dict(range=[0, 100]),
    height=350,
)
fig.show()

In [ ]:
# ── 10f. Volume ──────────────────────────────────────────────────────────────
vol_avg = df["Volume"].rolling(20).mean()

fig = go.Figure()
fig.add_trace(go.Bar(   x=df.index, y=df["Volume"], name="Volume",  marker_color="white", opacity=0.6))
fig.add_trace(go.Scatter(x=df.index, y=vol_avg,     name="20d Avg", line=dict(color="orange", width=1.5)))
fig.update_layout(title=f"{ticker} — Volume + 20-day Avg", height=350)
fig.show()

In [ ]:
# ── 10g. Fundamentals ─────────────────────────────────────────────────────────
ticker = TICKERS["RT"]
info   = yf.Ticker(ticker).info

pe    = info.get("trailingPE");     pe_str  = f"{pe:.2f}"    if pe    is not None else "N/A"
eps_g = info.get("earningsGrowth"); eps_str = f"{eps_g:.2%}" if eps_g is not None else "N/A"
de    = info.get("debtToEquity");   de_str  = f"{de:.2f}"    if de    is not None else "N/A"
cash  = info.get("totalCash");      c_str   = f"{cash:,.0f}" if cash  is not None else "N/A"
debt  = info.get("totalDebt");      d_str   = f"{debt:,.0f}" if debt  is not None else "N/A"
icr   = info.get("currentRatio");   icr_str = f"{icr:.2f}"   if icr   is not None else "N/A"
peg   = info.get("pegRatio");       peg_str = f"{peg:.2f}"   if peg   is not None else "N/A"
div_y = info.get("dividendYield");  div_str = f"{div_y:.2%}" if div_y is not None else "N/A"

print(f"  P/E Ratio        : {pe_str}")
print(f"  EPS Growth (TTM) : {eps_str}")
print(f"  Debt / Equity    : {de_str}")
print(f"  Total Cash       : ₹{c_str}")
print(f"  Total Debt       : ₹{d_str}")
print(f"  Current Ratio    : {icr_str}")
print(f"  PEG Ratio        : {peg_str}")
print(f"  Dividend Yield   : {div_str}")

### Observations

_TODO (RT): Write 3–5 paragraphs on what the technical and fundamental data reveals about LT.NS. Consider: trend direction, momentum signals (RSI / MACD), valuation (P/E, PEG), and financial health (D/E, cash position)._

## Consumer

## Section 11 — TITAN.NS | NS (Consumer)

In [ ]:
# ── 11a. Fetch ──────────────────────────────────────────────────────────────
ticker = TICKERS["NS"]
df     = fetch_stock(ticker)
print(f"{ticker} | {df.shape[0]} rows | {df.index[0].date()} → {df.index[-1].date()}")
df.tail()

In [ ]:
# ── 11b. Summary Statistics ──────────────────────────────────────────────────
print(f"Shape     : {df.shape}")
print(f"Date range: {df.index[0].date()} → {df.index[-1].date()}")
df.describe().round(2)

In [ ]:
# ── 11c. Price: Candlestick + EMA20 / EMA50 ────────────────────────────────
ema20 = EMAIndicator(close=df["Close"], window=20).ema_indicator()
ema50 = EMAIndicator(close=df["Close"], window=50).ema_indicator()

fig = make_subplots(rows=1, cols=1)
fig.add_trace(go.Candlestick(
    x=df.index, open=df["Open"], high=df["High"],
    low=df["Low"], close=df["Close"], name="OHLC",
))
fig.add_trace(go.Scatter(x=df.index, y=ema20, name="EMA20", line=dict(color="orange", width=1.5)))
fig.add_trace(go.Scatter(x=df.index, y=ema50, name="EMA50", line=dict(color="#4A90D9", width=1.5)))
fig.update_layout(
    title=f"{ticker} — Price + EMA20/50",
    xaxis_rangeslider_visible=False,
    height=500,
)
fig.show()

In [ ]:
# ── 11d. MACD (12/26/9) ────────────────────────────────────────────────────
macd_ind    = MACD(close=df["Close"], window_slow=26, window_fast=12, window_sign=9)
macd_line   = macd_ind.macd()
signal_line = macd_ind.macd_signal()
histogram   = macd_ind.macd_diff()

fig = make_subplots(
    rows=2, cols=1, shared_xaxes=True,
    row_heights=[0.6, 0.4],
    subplot_titles=[f"{ticker} — Close", "MACD (12/26/9)"],
)
fig.add_trace(go.Scatter(x=df.index, y=df["Close"], name="Close",  line=dict(color="white")),  row=1, col=1)
fig.add_trace(go.Scatter(x=df.index, y=macd_line,   name="MACD",   line=dict(color="cyan")),   row=2, col=1)
fig.add_trace(go.Scatter(x=df.index, y=signal_line, name="Signal", line=dict(color="orange")), row=2, col=1)
fig.add_trace(go.Bar(    x=df.index, y=histogram,   name="Hist",   marker_color="white"),       row=2, col=1)
fig.update_layout(height=600, xaxis_rangeslider_visible=False)
fig.show()

In [ ]:
# ── 11e. RSI (14) ───────────────────────────────────────────────────────────
rsi = RSIIndicator(close=df["Close"], window=14).rsi()

fig = go.Figure()
fig.add_trace(go.Scatter(x=df.index, y=rsi, name="RSI(14)", line=dict(color="mediumpurple", width=1.5)))
fig.add_hline(y=70, line_dash="dash", line_color="red",       annotation_text="Overbought (70)")
fig.add_hline(y=30, line_dash="dash", line_color="limegreen", annotation_text="Oversold (30)")
fig.update_layout(
    title=f"{ticker} — RSI (14)",
    yaxis=dict(range=[0, 100]),
    height=350,
)
fig.show()

In [ ]:
# ── 11f. Volume ──────────────────────────────────────────────────────────────
vol_avg = df["Volume"].rolling(20).mean()

fig = go.Figure()
fig.add_trace(go.Bar(   x=df.index, y=df["Volume"], name="Volume",  marker_color="white", opacity=0.6))
fig.add_trace(go.Scatter(x=df.index, y=vol_avg,     name="20d Avg", line=dict(color="orange", width=1.5)))
fig.update_layout(title=f"{ticker} — Volume + 20-day Avg", height=350)
fig.show()

In [ ]:
# ── 11g. Fundamentals ─────────────────────────────────────────────────────────
ticker = TICKERS["NS"]
info   = yf.Ticker(ticker).info

pe    = info.get("trailingPE");     pe_str  = f"{pe:.2f}"    if pe    is not None else "N/A"
eps_g = info.get("earningsGrowth"); eps_str = f"{eps_g:.2%}" if eps_g is not None else "N/A"
de    = info.get("debtToEquity");   de_str  = f"{de:.2f}"    if de    is not None else "N/A"
cash  = info.get("totalCash");      c_str   = f"{cash:,.0f}" if cash  is not None else "N/A"
debt  = info.get("totalDebt");      d_str   = f"{debt:,.0f}" if debt  is not None else "N/A"
icr   = info.get("currentRatio");   icr_str = f"{icr:.2f}"   if icr   is not None else "N/A"
peg   = info.get("pegRatio");       peg_str = f"{peg:.2f}"   if peg   is not None else "N/A"
div_y = info.get("dividendYield");  div_str = f"{div_y:.2%}" if div_y is not None else "N/A"

print(f"  P/E Ratio        : {pe_str}")
print(f"  EPS Growth (TTM) : {eps_str}")
print(f"  Debt / Equity    : {de_str}")
print(f"  Total Cash       : ₹{c_str}")
print(f"  Total Debt       : ₹{d_str}")
print(f"  Current Ratio    : {icr_str}")
print(f"  PEG Ratio        : {peg_str}")
print(f"  Dividend Yield   : {div_str}")

### Observations

_TODO (NS): Write 3–5 paragraphs on what the technical and fundamental data reveals about TITAN.NS. Consider: trend direction, momentum signals (RSI / MACD), valuation (P/E, PEG), and financial health (D/E, cash position)._

## Defence

## Section 12 — APOLLOMICRO.NS | SmS (Defence)

In [ ]:
# ── 12a. Fetch ──────────────────────────────────────────────────────────────
ticker = TICKERS["SmS"]
df     = fetch_stock(ticker)
print(f"{ticker} | {df.shape[0]} rows | {df.index[0].date()} → {df.index[-1].date()}")
df.tail()

In [ ]:
# ── 12b. Summary Statistics ──────────────────────────────────────────────────
print(f"Shape     : {df.shape}")
print(f"Date range: {df.index[0].date()} → {df.index[-1].date()}")
df.describe().round(2)

In [ ]:
# ── 12c. Price: Candlestick + EMA20 / EMA50 ────────────────────────────────
ema20 = EMAIndicator(close=df["Close"], window=20).ema_indicator()
ema50 = EMAIndicator(close=df["Close"], window=50).ema_indicator()

fig = make_subplots(rows=1, cols=1)
fig.add_trace(go.Candlestick(
    x=df.index, open=df["Open"], high=df["High"],
    low=df["Low"], close=df["Close"], name="OHLC",
))
fig.add_trace(go.Scatter(x=df.index, y=ema20, name="EMA20", line=dict(color="orange", width=1.5)))
fig.add_trace(go.Scatter(x=df.index, y=ema50, name="EMA50", line=dict(color="#4A90D9", width=1.5)))
fig.update_layout(
    title=f"{ticker} — Price + EMA20/50",
    xaxis_rangeslider_visible=False,
    height=500,
)
fig.show()

In [ ]:
# ── 12d. MACD (12/26/9) ────────────────────────────────────────────────────
macd_ind    = MACD(close=df["Close"], window_slow=26, window_fast=12, window_sign=9)
macd_line   = macd_ind.macd()
signal_line = macd_ind.macd_signal()
histogram   = macd_ind.macd_diff()

fig = make_subplots(
    rows=2, cols=1, shared_xaxes=True,
    row_heights=[0.6, 0.4],
    subplot_titles=[f"{ticker} — Close", "MACD (12/26/9)"],
)
fig.add_trace(go.Scatter(x=df.index, y=df["Close"], name="Close",  line=dict(color="white")),  row=1, col=1)
fig.add_trace(go.Scatter(x=df.index, y=macd_line,   name="MACD",   line=dict(color="cyan")),   row=2, col=1)
fig.add_trace(go.Scatter(x=df.index, y=signal_line, name="Signal", line=dict(color="orange")), row=2, col=1)
fig.add_trace(go.Bar(    x=df.index, y=histogram,   name="Hist",   marker_color="white"),       row=2, col=1)
fig.update_layout(height=600, xaxis_rangeslider_visible=False)
fig.show()

In [ ]:
# ── 12e. RSI (14) ───────────────────────────────────────────────────────────
rsi = RSIIndicator(close=df["Close"], window=14).rsi()

fig = go.Figure()
fig.add_trace(go.Scatter(x=df.index, y=rsi, name="RSI(14)", line=dict(color="mediumpurple", width=1.5)))
fig.add_hline(y=70, line_dash="dash", line_color="red",       annotation_text="Overbought (70)")
fig.add_hline(y=30, line_dash="dash", line_color="limegreen", annotation_text="Oversold (30)")
fig.update_layout(
    title=f"{ticker} — RSI (14)",
    yaxis=dict(range=[0, 100]),
    height=350,
)
fig.show()

In [ ]:
# ── 12f. Volume ──────────────────────────────────────────────────────────────
vol_avg = df["Volume"].rolling(20).mean()

fig = go.Figure()
fig.add_trace(go.Bar(   x=df.index, y=df["Volume"], name="Volume",  marker_color="white", opacity=0.6))
fig.add_trace(go.Scatter(x=df.index, y=vol_avg,     name="20d Avg", line=dict(color="orange", width=1.5)))
fig.update_layout(title=f"{ticker} — Volume + 20-day Avg", height=350)
fig.show()

In [ ]:
# ── 12g. Fundamentals ─────────────────────────────────────────────────────────
ticker = TICKERS["SmS"]
info   = yf.Ticker(ticker).info

pe    = info.get("trailingPE");     pe_str  = f"{pe:.2f}"    if pe    is not None else "N/A"
eps_g = info.get("earningsGrowth"); eps_str = f"{eps_g:.2%}" if eps_g is not None else "N/A"
de    = info.get("debtToEquity");   de_str  = f"{de:.2f}"    if de    is not None else "N/A"
cash  = info.get("totalCash");      c_str   = f"{cash:,.0f}" if cash  is not None else "N/A"
debt  = info.get("totalDebt");      d_str   = f"{debt:,.0f}" if debt  is not None else "N/A"
icr   = info.get("currentRatio");   icr_str = f"{icr:.2f}"   if icr   is not None else "N/A"
peg   = info.get("pegRatio");       peg_str = f"{peg:.2f}"   if peg   is not None else "N/A"
div_y = info.get("dividendYield");  div_str = f"{div_y:.2%}" if div_y is not None else "N/A"

print(f"  P/E Ratio        : {pe_str}")
print(f"  EPS Growth (TTM) : {eps_str}")
print(f"  Debt / Equity    : {de_str}")
print(f"  Total Cash       : ₹{c_str}")
print(f"  Total Debt       : ₹{d_str}")
print(f"  Current Ratio    : {icr_str}")
print(f"  PEG Ratio        : {peg_str}")
print(f"  Dividend Yield   : {div_str}")

### Observations

_TODO (SmS): Write 3–5 paragraphs on what the technical and fundamental data reveals about APOLLOMICRO.NS. Consider: trend direction, momentum signals (RSI / MACD), valuation (P/E, PEG), and financial health (D/E, cash position)._

## Section 13 — Cross-Sector Correlation Heatmap | RS + GT

In [ ]:
import logging
import numpy as np
import pandas as pd
import plotly.express as px


logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

def generate_correlation_matrix(tickers_dict: dict, period: str = "1y", interval: str = "1d") -> None:
    """
    Defensively fetches close prices for all mapped NIFTY50 tickers, calculates
    aligned daily log returns, and renders an interactive cross-sector correlation heatmap.
    """
    close_series_map = {}
    logger.info("Executing defensive correlation data gathering matrix...")

    for handle, ticker in tickers_dict.items():
        try:
            df = fetch_stock(ticker, period=period, interval=interval)
            if df is not None and not df.empty and "Close" in df.columns:
                clean_handle = f"{handle} ({ticker.replace('.NS', '')})"
                close_series_map[clean_handle] = df["Close"]
            else:
                logger.warning(f"Ticker {ticker} returned empty or missing 'Close' vector. Skipping matrix inclusion.")
        except Exception as e:
            logger.error(f"Critical failure processing data array for handle {handle} ({ticker}): {str(e)}")
            continue

    if len(close_series_map) < 2:
        raise ValueError("Insufficient data fields captured. A minimum of 2 valid asset series are required to map a matrix.")

    price_df = pd.DataFrame(close_series_map).dropna(how="any")
    logger.info(f"Time-series aligned successfully. Dimension: {price_df.shape[0]} trading days x {price_df.shape[1]} symbols.")

    log_returns = np.log(price_df / price_df.shift(1)).dropna()
    correlation_matrix = log_returns.corr(method="pearson")

    fig = px.imshow(
        correlation_matrix,
        text_auto=".2f",
        color_continuous_scale="RdBu_r",
        zmin=-1.0, zmax=1.0,
        labels=dict(x="Asset Identifier", y="Asset Identifier", color="Pearson r"),
        title="QuantIQ | Core NIFTY50 Matrix - 1 Year Daily Return Correlation Profile"
    )

    fig.update_layout(
        height=700, width=750,
        coloraxis_colorbar=dict(
            title="Correlation Profile",
            thicknessmode="pixels", thickness=15,
            lenmode="fraction",
            len=0.8
        ),
        margin=dict(l=50, r=50, b=50, t=80)
    )
    fig.show()

### Observations

_TODO (RS + GT): Write 3–5 paragraphs on correlation patterns — which sectors move together, which are uncorrelated, and what this implies for portfolio diversification._